# Assignment 5 · Task 2 — Part A: Human Matting Model

**Architecture:** U-Net (encoder–decoder with skip connections) trained from scratch on AISegment  
**Loss:** `0.5 × L1 + 0.5 × Dice`  
**Target:** Val IoU ≥ 0.85  
**Dataset:** [AISegment Matting Human Datasets](https://www.kaggle.com/datasets/laurentmih/aisegmentcom-matting-human-datasets)

---
### How this notebook works
1. Checks GPU / environment  
2. Clones your GitHub repo (all `.py` + `config.yaml` already there)  
3. Installs `requirements.txt`  
4. Verifies the dataset is mounted  
5. Patches `config.yaml` with Kaggle-specific settings (workers, paths)  
6. Trains the model (`train.py`)  
7. Evaluates on test split (`evaluate.py`)  
8. Plots training curves + sample predictions (`visualise.py`)  
9. Zips everything for download  

> **Before running:** set `GITHUB_REPO` in Cell 3 to your repo URL.

## 1 · Environment Check

In [ ]:
import subprocess, sys, os, shutil, time
from pathlib import Path

# ── GPU ────────────────────────────────────────────────────────────────────
print("=" * 60)
print("GPU / CUDA")
print("=" * 60)
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

import torch
print(f"\nPyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")
    print(f"VRAM (total)    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Disk ───────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Disk space")
print("=" * 60)
!df -h /kaggle/working

## 2 · Clone GitHub Repo

**Set `GITHUB_REPO` to your repository URL.**  
Example: `https://github.com/YourUsername/CV-AS-5-TASK2`

In [ ]:
# ── ✏️  EDIT THIS ──────────────────────────────────────────────────────────
GITHUB_REPO   = "https://github.com/YOUR_USERNAME/CV-AS-5-TASK2"
# ──────────────────────────────────────────────────────────────────────────

REPO_DIR = Path("/kaggle/working/repo")

if REPO_DIR.exists():
    print("Repo already cloned — pulling latest …")
    !git -C {REPO_DIR} pull
else:
    print(f"Cloning {GITHUB_REPO} …")
    !git clone --depth 1 {GITHUB_REPO} {REPO_DIR}

print("\nRepo contents:")
for p in sorted(REPO_DIR.iterdir()):
    print(f"  {p.name}")

# Add repo to Python path so we can import model, dataset, etc.
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("\n✓ Repo ready.")

## 3 · Install Dependencies

In [ ]:
req_path = REPO_DIR / "requirements.txt"

if req_path.exists():
    print("Installing from requirements.txt …")
    !pip install -q -r {req_path}
else:
    # Fallback: install individually
    print("requirements.txt not found — installing core packages …")
    !pip install -q tqdm pyyaml

# Verify key imports
import yaml, tqdm
print(f"PyYAML {yaml.__version__} | tqdm {tqdm.__version__}")
print("✓ All dependencies ready.")

## 4 · Verify Dataset Mount

In [ ]:
DATASET_ROOT = Path("/kaggle/input/aisegmentcom-matting-human-datasets")

assert DATASET_ROOT.exists(), (
    f"Dataset not found at {DATASET_ROOT}.\n"
    "Add the AISegment dataset to this notebook via:\n"
    "  Notebook → Data → Add Dataset → Search 'aisegmentcom-matting-human-datasets'"
)

clip_imgs = list(DATASET_ROOT.glob("clip_img/**/*_clip.jpg"))
mattes    = list(DATASET_ROOT.glob("matting/**/*.png"))

print(f"Dataset root : {DATASET_ROOT}")
print(f"Clip images  : {len(clip_imgs):,}")
print(f"Matte PNGs   : {len(mattes):,}")

if len(clip_imgs) == 0:
    # Try alternate glob
    clip_imgs = list(DATASET_ROOT.glob("clip_img/**/*.jpg"))
    print(f"(alt glob)   : {len(clip_imgs):,} jpg images")

assert len(clip_imgs) > 0, "No clip images found — check dataset structure."
assert len(mattes)    > 0, "No mattes found — check dataset structure."

# Show a sample pair
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

sample_img   = Image.open(clip_imgs[0]).convert("RGB")

def _resolve_matte(img_path):
    parts = list(img_path.parts)
    parts[parts.index("clip_img")] = "matting"
    stem = img_path.stem.replace("_clip", "")
    parts[-1] = stem + ".png"
    return Path(*parts)

sample_matte_path = _resolve_matte(clip_imgs[0])
sample_matte = Image.open(sample_matte_path) if sample_matte_path.exists() else Image.new("L", sample_img.size, 128)
if sample_matte.mode == "RGBA": sample_matte = sample_matte.split()[3]
else: sample_matte = sample_matte.convert("L")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(np.array(sample_img));  axes[0].set_title("Sample RGB");   axes[0].axis("off")
axes[1].imshow(np.array(sample_matte), cmap="gray"); axes[1].set_title("Sample Matte"); axes[1].axis("off")
plt.suptitle("Dataset sample — verify these look correct", fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/dataset_sample.png", dpi=120, bbox_inches="tight")
plt.show()
print("✓ Dataset verified.")

## 5 · Configure for Kaggle

We write a `kaggle_config.yaml` that overrides a few settings optimised for the T4 GPU (4 workers, batch 16, 25 epochs).  
All paths already point to `/kaggle/input` and `/kaggle/working` in the base `config.yaml`, so only worker count changes.

In [ ]:
import yaml, copy

# Load the repo's config.yaml
base_cfg_path = REPO_DIR / "config.yaml"
with open(base_cfg_path) as fh:
    cfg = yaml.safe_load(fh)

# ── Kaggle overrides ──────────────────────────────────────────────────────
cfg["data"]["dataset_root"] = str(DATASET_ROOT)
cfg["data"]["output_dir"]   = "/kaggle/working/outputs"
cfg["matting"]["weights_path"] = "/kaggle/working/matting_weights.pth"
cfg["matting"]["num_workers"]  = 4      # T4 instance supports this
cfg["matting"]["batch_size"]   = 16     # safe for 16 GB T4 VRAM at 320×320
cfg["matting"]["epochs"]       = 25     # ≥ 20 as required
cfg["matting"]["train_size"]   = 5000
cfg["matting"]["val_size"]     = 500
cfg["matting"]["test_size"]    = 500

# ── Save override config ──────────────────────────────────────────────────
KAGGLE_CFG = Path("/kaggle/working/kaggle_config.yaml")
with open(KAGGLE_CFG, "w") as fh:
    yaml.dump(cfg, fh, default_flow_style=False, sort_keys=False)

print("kaggle_config.yaml written:")
print("-" * 50)
with open(KAGGLE_CFG) as fh:
    print(fh.read())

# Create output directory
Path("/kaggle/working/outputs").mkdir(exist_ok=True)
print("✓ Config ready.")

## 6 · Smoke-Test the Model

Verify both architectures produce correct output shapes before touching the dataset.

In [ ]:
import os; os.chdir(REPO_DIR)   # so that relative imports (model, dataset) work
!python model.py

## 7 · Train the Matting Model

Runs `train.py` with the Kaggle config. Progress is streamed in real-time.

Expected wall-clock time on a **T4 GPU**: ~2–4 min/epoch → ~50–100 min total for 25 epochs.

In [ ]:
import subprocess, sys

train_cmd = [
    sys.executable, str(REPO_DIR / "train.py"),
    "--config", str(KAGGLE_CFG),
]

print("Command:", " ".join(train_cmd))
print("=" * 60)

# Stream output line-by-line so progress is visible in the notebook
proc = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, cwd=str(REPO_DIR),
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f"train.py exited with code {proc.returncode}")
print("\n✓ Training complete.")

## 8 · Full Evaluation on Test Split

In [ ]:
eval_cmd = [
    sys.executable, str(REPO_DIR / "evaluate.py"),
    "--weights", "/kaggle/working/matting_weights.pth",
    "--config",  str(KAGGLE_CFG),
    "--split",   "test",
]

print("Command:", " ".join(eval_cmd))
print("=" * 60)

proc = subprocess.Popen(
    eval_cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, cwd=str(REPO_DIR),
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f"evaluate.py exited with code {proc.returncode}")
print("\n✓ Evaluation complete.")

## 9 · Plot Training Curves

In [ ]:
log_csv = Path("/kaggle/working/outputs/matting_train_log.csv")

assert log_csv.exists(), f"Training log not found: {log_csv}"

curves_cmd = [
    sys.executable, str(REPO_DIR / "visualise.py"),
    "curves",
    "--log", str(log_csv),
    "--out", "/kaggle/working/outputs/training_curves.png",
]
proc = subprocess.Popen(
    curves_cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, cwd=str(REPO_DIR),
)
for line in proc.stdout:
    print(line, end="")
proc.wait()

# Display in notebook
from IPython.display import Image as IPyImage, display
display(IPyImage("/kaggle/working/outputs/training_curves.png"))

## 10 · Sample Predictions Grid

In [ ]:
preds_cmd = [
    sys.executable, str(REPO_DIR / "visualise.py"),
    "predictions",
    "--weights", "/kaggle/working/matting_weights.pth",
    "--config",  str(KAGGLE_CFG),
    "--n",       "8",
    "--out",     "/kaggle/working/outputs/sample_predictions.png",
]
proc = subprocess.Popen(
    preds_cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, cwd=str(REPO_DIR),
)
for line in proc.stdout:
    print(line, end="")
proc.wait()

from IPython.display import Image as IPyImage, display
display(IPyImage("/kaggle/working/outputs/sample_predictions.png"))

## 11 · Display Training Log Table

In [ ]:
import csv
import pandas as pd

log_csv = Path("/kaggle/working/outputs/matting_train_log.csv")
df = pd.read_csv(log_csv)

# Style the table
best_iou_row = df["val_iou"].idxmax()

def highlight_best(s):
    is_best = s.index == best_iou_row
    return ["background-color: #d4edda; font-weight: bold" if v else "" for v in is_best]

print(f"Best val IoU = {df['val_iou'].max():.4f}  (epoch {df.loc[best_iou_row, 'epoch']})")
print(f"Final epoch  = {df['epoch'].max()}")
print()

display(
    df.style
      .apply(highlight_best, axis=0)
      .format({
          "train_loss": "{:.4f}",
          "val_loss":   "{:.4f}",
          "val_iou":    "{:.4f}",
          "lr":         "{:.2e}",
      })
      .set_caption("Training log (green = best val IoU)")
)

## 12 · Package Outputs for Download

Zips the checkpoint, logs, and visualisations into a single file visible in the Kaggle output panel.

In [ ]:
import shutil, os

outputs = Path("/kaggle/working/outputs")
weights = Path("/kaggle/working/matting_weights.pth")

# Copy checkpoint into outputs folder for single-zip convenience
if weights.exists():
    shutil.copy(weights, outputs / "matting_weights.pth")

zip_base = "/kaggle/working/part_a_results"
shutil.make_archive(zip_base, "zip", outputs)

zip_path = zip_base + ".zip"
size_mb  = os.path.getsize(zip_path) / 1e6

print(f"\n✓ Output archive: {zip_path}  ({size_mb:.1f} MB)")
print("\nContents:")
for p in sorted(outputs.iterdir()):
    sz = os.path.getsize(p) / 1e6
    print(f"  {p.name:<40}  {sz:.2f} MB")

print("\nDownload from:  Kaggle notebook → Output → part_a_results.zip")

---
## ✅ Done

| File | Description |
|---|---|
| `matting_weights.pth` | Best model checkpoint |
| `matting_train_log.csv` | Epoch-by-epoch loss & IoU |
| `training_curves.png` | Loss + IoU curve plots |
| `sample_predictions.png` | Input / GT / Predicted matte grid |
| `dataset_sample.png` | Sanity-check dataset sample |

All files are in `/kaggle/working/outputs/` and bundled in **`part_a_results.zip`**.